# Declaring a ReactionSystem

Canonical "here are the building blocks of chemistry" demo for the direct-API
layer. Demonstrates each reaction kind the framework knows about and how a
`ReactionSystem` routes them:

- **`KineticReaction`** -- a single integrated reaction with a callable rate
  law. Built here via `ReactionBuilder.aerobic_growth`, which derives the
  stoichiometric coefficients from substrate/biomass elemental formulas and a
  yield.
- **`EquilibriumReaction` (single-phase)** -- an algebraic constraint with
  `log_K`. Acetic acid dissociation (`AceticAcid <-> Acetate- + H+`) is the
  example here.
- **`EquilibriumReaction` (cross-phase)** -- a partition declaration that
  spans two phase keys. Gives the gas-liquid link its molecular-form mapping
  (`{"CO2": "CO2"}`) without the user wiring it by hand. `log_K` is omitted --
  the partition constant lives on the link (Henry's law), not the reaction.

The module exposes three factory functions so other tutorials
([`../D2C_workshop/raw_construction.py`](../D2C_workshop/raw_construction.py))
can declare the same chemistry independently (a by-value copy, not an
import -- each tutorial folder stays self-sufficient).

In [1]:
from PyOMES.chemistry import Species
from PyOMES.chemistry.common_species import H_plus
from PyOMES.reactions import (
    EquilibriumReaction,
    KineticReaction,
    ReactionBuilder,
    ReactionSystem,
    StoichiometryEntry,
)

print("Imports OK")

Imports OK


## Species

Acetic acid: HA (neutral) and its conjugate base. Acetate- isn't in
`common_species` (which only holds universal inorganics), so it's declared
here alongside the dissociation it participates in. `CO2` is declared
locally (rather than imported from `common_species`) so its MW matches the
value `ReactionBuilder.aerobic_growth` uses when it constructs its own
internal CO2 -- without the match, the CV's species-consistency check raises
on the tiny MW disagreement (44.009 vs `common_species`' 44.01).

In [2]:
ACETIC_ACID = Species(
    id="AceticAcid", atoms={"C": 2, "H": 4, "O": 2}, charge=0, MW=60.052,
)
ACETATE_MINUS = Species(
    id="Acetate-", atoms={"C": 2, "H": 3, "O": 2}, charge=-1, MW=59.044,
)
CO2 = Species(id="CO2", atoms={"C": 1, "O": 2}, charge=0, MW=44.009)

# Biomass: a CHO pseudo-molecule "Yeast" with the same elemental
# composition the FermenterBuilder uses by default.
YEAST = Species(
    id="Yeast", atoms={"C": 1, "H": 1.61, "O": 0.56}, charge=0, MW=24.626,
)

print("Species declared:", ACETIC_ACID.id, ACETATE_MINUS.id, CO2.id, YEAST.id)

Species declared: AceticAcid Acetate- CO2 Yeast


## `KineticReaction` -- aerobic growth on acetic acid

Stoichiometry is derived from elemental balance -- the user supplies only
the substrate/biomass formulas and the yield; CO2, O2 and H2O coefficients
fall out of the balance. The rate function evaluates Monod growth on the
substrate mass concentration and returns extensive substrate consumption
(mol/h), matching the convention `ReactionBuilder.aerobic_growth` expects.

In [3]:
def make_aerobic_growth_on_acetate(
    mu_max_per_h: float = 0.5,
    Ks_g_per_L: float = 5e-3,
    yield_gX_gS: float = 0.36,
) -> KineticReaction:
    MW_S = float(ACETIC_ACID.MW)
    MW_X = float(YEAST.MW)
    Y = float(yield_gX_gS)

    def rate_fn(env):
        # env.concentrations are mol/L; convert substrate to g/L for
        # Monod, biomass to g/L for the specific-rate-to-extensive-rate
        # conversion.
        C_S = env.concentrations.get("AceticAcid", 0.0)
        C_X = env.concentrations.get("Yeast", 0.0)
        S_gL = C_S * MW_S
        X_gL = C_X * MW_X
        if X_gL <= 1e-30 or S_gL <= 0.0:
            return 0.0
        mu = mu_max_per_h * S_gL / (Ks_g_per_L + S_gL)
        # Extensive substrate consumption (mol/h): (mu/Y) * X * V / MW_S
        return (mu / Y) * X_gL / MW_S * env.V_L

    return ReactionBuilder.aerobic_growth(
        substrate_id=ACETIC_ACID.id,
        substrate_atoms=dict(ACETIC_ACID.atoms),
        MW_substrate=MW_S,
        biomass_id=YEAST.id,
        biomass_atoms=dict(YEAST.atoms),
        MW_biomass=MW_X,
        yield_gX_gS=Y,
        rate_fn=rate_fn,
        balance="CHO",
        label="growth_on_AceticAcid",
    )

print(make_aerobic_growth_on_acetate())

KineticReaction(-1 AceticAcid, -1.01 O2, +0.878 Yeast, +1.12 CO2, +1.29 H2O [growth_on_AceticAcid])


## `EquilibriumReaction` (single-phase) -- acetic acid dissociation

`HA <-> A- + H+`. `log_K = -pKa` by the products/reactants convention. The
speciation engine consumes this as a single-phase algebraic constraint and
writes back `n_mol["H+"]`, `n_mol["Acetate-"]`, and `n_mol["AceticAcid"]`
(the molecular form) after each solve.

In [4]:
def make_acetate_dissociation(pKa: float = 4.756) -> EquilibriumReaction:
    return EquilibriumReaction(
        stoichiometry=[
            StoichiometryEntry(species=ACETIC_ACID, phase="liquid", coefficient=-1.0),
            StoichiometryEntry(species=ACETATE_MINUS, phase="liquid", coefficient=+1.0),
            StoichiometryEntry(species=H_plus, phase="liquid", coefficient=+1.0),
        ],
        log_K=-float(pKa),
        balance_elements=("C", "H", "O"),
        label="eq_AceticAcid",
    )

print(make_acetate_dissociation())

EquilibriumReaction(-1 AceticAcid, +1 Acetate-, +1 H+ [eq_AceticAcid])


## `EquilibriumReaction` (cross-phase) -- CO2 partition

`CO2(gas) <-> CO2aq(liquid)`. A cross-phase `EquilibriumReaction` is a
*partition declaration*, not a thermodynamic constraint -- it tells a
`KineticGasLiquidLink` which liquid-phase species id corresponds to the
molecular form of the gas-phase species. `log_K` is omitted because the
partition constant lives on the link (Henry's law), not on the reaction.
After this reaction is attached to a `ReactionSystem` and the system is
attached to a `ControlVolume` that carries a gas-liquid link, the link's
`derive_speciation_keys()` call populates
`link.speciation_keys["CO2"] = "CO2"` from this declaration (phase-agnostic
id: gas and dissolved CO2 share the same `Species`).

In [5]:
def make_co2_partition() -> EquilibriumReaction:
    return EquilibriumReaction(
        stoichiometry=[
            StoichiometryEntry(species=CO2, phase="gas", coefficient=-1.0),
            StoichiometryEntry(species=CO2, phase="liquid", coefficient=+1.0),
        ],
        balance_elements=("C", "O"),
        label="partition_CO2",
    )

print(make_co2_partition())

EquilibriumReaction(-1 CO2, +1 CO2 [partition_CO2])


## Assembling the `ReactionSystem`

The system pre-buckets reactions by type at construction. After this call,
four properties expose the type-specific projections:
`kinetic_reactions`, `single_phase_equilibria`, `cross_phase_equilibria`,
and `blackbox_models` (empty here -- see the [`fba/`](fba/) sibling demos
for that bucket).

In [6]:
def build_reaction_system() -> ReactionSystem:
    return ReactionSystem(
        [
            make_aerobic_growth_on_acetate(),
            make_acetate_dissociation(),
            make_co2_partition(),
        ],
        label="reaction_system_demo",
    )


def _format_stoichiometry(rxn) -> str:
    parts = []
    for entry in rxn.stoichiometry:
        parts.append(
            f"{entry.coefficient:+.3g} {entry.species.id} ({entry.phase})"
        )
    return "  " + " ".join(parts)


system = build_reaction_system()

print("3 reactions declared; the system pre-buckets them by type at construction.")
print()
print(f"  {system!r}")
print()

print("Kinetic reactions ({}):".format(len(system.kinetic_reactions)))
for rxn in system.kinetic_reactions:
    print(f"  [{rxn.label}]")
    print(_format_stoichiometry(rxn))
print()

print(
    "Single-phase equilibria ({}):".format(
        len(system.single_phase_equilibria)
    )
)
for rxn in system.single_phase_equilibria:
    print(f"  [{rxn.label}]  log_K = {rxn.log_K:+.4g}")
    print(_format_stoichiometry(rxn))
print()

print(
    "Cross-phase equilibria ({}):".format(
        len(system.cross_phase_equilibria)
    )
)
for rxn in system.cross_phase_equilibria:
    log_K_str = (
        f"{rxn.log_K:+.4g}" if rxn.log_K is not None
        else "n/a (partition declaration; constant lives on the link)"
    )
    print(f"  [{rxn.label}]  log_K = {log_K_str}")
    print(_format_stoichiometry(rxn))
print()

print("Black-box models ({}):".format(len(system.blackbox_models)))
if not system.blackbox_models:
    print("  (none -- see fba/ for BlackBoxReactionModel examples)")
print()

print(
    "Aggregated species ({}): {}".format(
        len(system.species_ids), ", ".join(system.species_ids)
    )
)
print(
    "Aggregated phases ({}): {}".format(
        len(system.phases), ", ".join(system.phases)
    )
)

3 reactions declared; the system pre-buckets them by type at construction.

  ReactionSystem(1 kinetic, 1 single-phase eq, 1 cross-phase eq, 0 precipitation eq, 0 black-box, solver='charge_balance' [reaction_system_demo])

Kinetic reactions (1):
  [growth_on_AceticAcid]
  -1 AceticAcid (liquid) -1.01 O2 (liquid) +0.878 Yeast (liquid) +1.12 CO2 (liquid) +1.29 H2O (liquid)

Single-phase equilibria (1):
  [eq_AceticAcid]  log_K = -4.756
  -1 AceticAcid (liquid) +1 Acetate- (liquid) +1 H+ (liquid)

Cross-phase equilibria (1):
  [partition_CO2]  log_K = n/a (partition declaration; constant lives on the link)
  -1 CO2 (gas) +1 CO2 (liquid)

Black-box models (0):
  (none -- see fba/ for BlackBoxReactionModel examples)

Aggregated species (7): Acetate-, AceticAcid, CO2, H+, H2O, O2, Yeast
Aggregated phases (2): gas, liquid
